# 06 — Preprocessing

Window each trial into fixed-length tiles, normalise, resize, and save as
`.npy` under `outputs/preprocessed/`. Training reads from the manifest CSV
this notebook produces.

Default config: 3 s windows with 1.5 s stride, log + Z-score, resized to
224×224 (foot + torso → 2 channels).


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.io as sio

plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.figsize"] = (10, 4)

from src.preprocessing import PreprocConfig, preprocess_dataset, process_trial
from src.data_loader import iter_trials


## 1. Smoke test on a single trial

In [ ]:
cfg = PreprocConfig(window_s=3.0, stride_s=1.5, out_size=(224, 224))
out_root = ROOT / "outputs" / "preprocessed"
out_root.mkdir(parents=True, exist_ok=True)

trial = next(iter_trials())
rows = process_trial(trial, cfg, out_root)
print(f"{trial.subject_id}/{trial.test}/{trial.trial}: produced {len(rows)} windows")
rows[:3]


## 2. Visualise one preprocessed window

In [ ]:
sample_path = out_root / rows[0]["npy_path"]
arr = np.load(sample_path)
print(f"shape={arr.shape}  dtype={arr.dtype}  range=[{arr.min():.2f}, {arr.max():.2f}]")

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, ch, name in zip(axes, arr, ["foot", "torso"]):
    ax.imshow(ch, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(f"{name}  (normalised)")
plt.tight_layout(); plt.show()


## 3. Limited dry-run (10 trials)

Don't run the full pass yet — confirm shapes and counts on a small sample first.

In [ ]:
manifest_small = preprocess_dataset(cfg=cfg, out_root=out_root, limit=10)
print(manifest_small[["subject_id", "group", "test", "trial", "window_idx", "t_start_s", "t_end_s"]].head())
print(f"\nwindows per trial: {manifest_small.groupby(['subject_id', 'test', 'trial']).size().describe()}")


## 4. Full preprocessing pass

This loads every `.mat` (~25–40 min on a Mac SSD). Disk usage: roughly
N × 2ch × 224 × 224 × 4 bytes ≈ 400 KB per window. Expect ~10–30 windows
per trial × 348 trials → ~3–10 GB of .npy cache.

Pass the excluded paths from Notebook 04 to skip flagged files. Set
`RUN_FULL = True` to execute.


In [ ]:
RUN_FULL = False

if RUN_FULL:
    excluded_path = ROOT / "excluded_files.csv"
    excluded = []
    if excluded_path.exists():
        excluded = pd.read_csv(excluded_path)["path"].tolist()
    print(f"excluding {len(excluded)} flagged files")
    manifest = preprocess_dataset(cfg=cfg, out_root=out_root, excluded_paths=excluded)
else:
    print("set RUN_FULL = True to run the full pass")
    manifest = manifest_small


## 5. Manifest sanity check

In [ ]:
print(f"total windows: {len(manifest)}")
print("\nwindows per group:")
print(manifest["group"].value_counts())
print("\nwindows per subject (head):")
print(manifest.groupby(["subject_id", "group"]).size().head(10))


### Next steps

- Notebook 07 reads `manifest.csv` and trains SVM / Random Forest baselines.
- Notebook 08 trains the CNN on the same manifest.
- If you want to try a different window length or representation, change the `PreprocConfig` here and rerun. Existing .npy files in `outputs/preprocessed/<group>/` get overwritten when filenames collide.
